In [15]:
#Blessing Anoroh
# Project 3
#DATA 620 - Web Analytics
# April 6, 2026 (Due)


#Instructions
#Using any of the three classifiers described in chapter 6 of Natural Language Processing with Python, and any features you can think of, build the best name gender classifier you can.
#Begin by splitting the Names Corpus into three subsets: 500 words for the test set, 500 words for the dev-test set, and the remaining 6900 words for the training set.
#Then, starting with the example name gender classifier, make incremental improvements.
#Use the dev-test set to check your progress. Once you are satisfied with your classifier, check its final performance on the test set.
#How does the performance on the test set compare to the performance on the dev-test set? Is this what you'd expect?


**Overview:**

This project involved building a gender classifier using the Names Corpus and improving it step by step by adding more features. Each version of the model was evaluated on a dev-test set, and the best model was selected and tested on a separate test set to measure its final performance.

In [16]:
import random
import nltk
from nltk.corpus import names
from nltk.classify import accuracy

# load the names dataset from NLTK (male and female names) to use it for training and testing
nltk.download('names')

# I combined male and female names into one labeled list
name_data = ([(person_name, 'male') for person_name in names.words('male.txt')] +
             [(person_name, 'female') for person_name in names.words('female.txt')])

random.seed(42)
random.shuffle(name_data)

# split data into train, dev-test, and final test sets
final_test = name_data[:500]
development_test = name_data[500:1000]
training_data = name_data[1000:]

print("Training items:", len(training_data))
print("Development-test items:", len(development_test))
print("Final test items:", len(final_test))

Training items: 6944
Development-test items: 500
Final test items: 500


[nltk_data] Downloading package names to /root/nltk_data...
[nltk_data]   Package names is already up-to-date!


In [17]:
#  defined basic feature function using only the last letter of the name
def extract_features_v1(person_name):
    person_name = person_name.lower()  # convert name to lowercase for consistency
    return {
        'ending_letter': person_name[-1]  # use the last character as the feature
    }

# created training data by extracting features from each name
train_examples_1 = [(extract_features_v1(n), label) for (n, label) in training_data]

# created dev-test data to evaluate how the model is performing
dev_examples_1 = [(extract_features_v1(n), label) for (n, label) in development_test]

# train a Naive Bayes classifier using the training data
model_1 = nltk.NaiveBayesClassifier.train(train_examples_1)

# print the accuracy of the baseline model on the dev-test set
print("Baseline accuracy on dev-test set:", accuracy(model_1, dev_examples_1))

Baseline accuracy on dev-test set: 0.754


The baseline model, which only used the last letter of each name, achieved an accuracy of 0.754 on the dev-test set. This shows that even a simple feature can capture some patterns in names, but there is still room for improvement.

In [18]:
# improved feature function with more information from the name
def extract_features_v2(person_name):
    person_name = person_name.lower()  # make everything lowercase for consistency
    vowel_letters = 'aeiou'  # define vowels to help count patterns

    return {
        'starting_letter': person_name[0],  # first letter of the name
        'ending_letter': person_name[-1],  # last letter of the name
        'ending_two': person_name[-2:] if len(person_name) > 1 else person_name,  # last 2 letters
        'ending_three': person_name[-3:] if len(person_name) > 2 else person_name,  # last 3 letters
        'name_size': len(person_name),  # total length of the name
        'total_vowels': sum(1 for ch in person_name if ch in vowel_letters),  # count how many vowels
        'ends_with_vowel': person_name[-1] in vowel_letters  # check if name ends in a vowel
    }

# build training set using the new feature function
train_examples_2 = [(extract_features_v2(n), label) for (n, label) in training_data]

# build dev-test set to evaluate improvements
dev_examples_2 = [(extract_features_v2(n), label) for (n, label) in development_test]

# train the classifier with the updated features
model_2 = nltk.NaiveBayesClassifier.train(train_examples_2)

# print accuracy to see if this version improved over the baseline
print("Improved accuracy on dev-test set:", accuracy(model_2, dev_examples_2))

Improved accuracy on dev-test set: 0.8


The improved model achieved an accuracy of 0.80 on the dev-test set. This is higher than the baseline accuracy, showing that adding features such as the first letter, multiple ending letters, and vowel counts allows the model to capture more patterns in names and improve performance.

In [19]:
# final feature function with more detailed patterns from the name
def extract_features_v3(person_name):
    person_name = person_name.lower()  # convert to lowercase to keep things consistent
    vowel_letters = 'aeiou'  # list of vowels for counting and checks

    return {
        'starting_letter': person_name[0],  # first letter of the name
        'ending_letter': person_name[-1],  # last letter of the name
        'first_two': person_name[:2],  # first 2 letters  (short prefix)
        'first_three': person_name[:3],  # first 3 letters (long prefix)
        'last_two': person_name[-2:] if len(person_name) > 1 else person_name,  # last 2 letters
        'last_three': person_name[-3:] if len(person_name) > 2 else person_name,  # last 3 letters
        'name_size': len(person_name),  # total number of characters in the name
        'vowel_total': sum(1 for ch in person_name if ch in vowel_letters),  # count vowels in the name
        'consonant_total': sum(1 for ch in person_name if ch.isalpha() and ch not in vowel_letters),  # count consonants
        'starts_with_vowel': person_name[0] in vowel_letters,  # check if name starts with a vowel
        'ends_with_vowel': person_name[-1] in vowel_letters  # check if name ends with a vowel
    }

# build training data using the final feature set
train_examples_3 = [(extract_features_v3(n), label) for (n, label) in training_data]

# build dev-test data to evaluate performance
dev_examples_3 = [(extract_features_v3(n), label) for (n, label) in development_test]

# train the classifier using the most detailed features
model_3 = nltk.NaiveBayesClassifier.train(train_examples_3)

# print accuracy to confirm this is the best performing model so far
print("Best dev-test accuracy:", accuracy(model_3, dev_examples_3))

Best dev-test accuracy: 0.842


The final model achieved an accuracy of 0.842 on the dev-test set. This was the highest accuracy among the three versions, which suggests that the added features helped the classifier recognize more detailed patterns in names.

In [20]:
# use the final feature set to prepare the test data
final_examples = [(extract_features_v3(n), label) for (n, label) in final_test]

# check the model’s accuracy on completely new data
print("Final test accuracy:", accuracy(model_3, final_examples))

Final test accuracy: 0.794


The final model achieved an accuracy of 0.794 on the test set. This is lower than the dev-test accuracy of 0.842, which suggests the model performed a little better on the development data than on completely unseen data.

In [21]:
# create a list to store misclassified names
mistakes = []

# go through each name in the dev-test set
for person_name, actual_label in development_test:
    # predict the label using the trained model
    predicted_label = model_3.classify(extract_features_v3(person_name))

    # check if the prediction is incorrect
    if predicted_label != actual_label:
        # save the actual label, predicted label, and the name
        mistakes.append((actual_label, predicted_label, person_name))

# print a sample of the dev-test errors to see where the model struggles
print("Some dev-test errors:")
for row in mistakes[:20]:
    print(row)

Some dev-test errors:
('female', 'male', 'Cherey')
('female', 'male', 'Marlo')
('female', 'male', 'Hildagard')
('female', 'male', 'Melicent')
('female', 'male', 'Moll')
('male', 'female', 'Georgie')
('female', 'male', 'Charil')
('male', 'female', 'Etienne')
('female', 'male', 'Yehudit')
('male', 'female', 'Merry')
('female', 'male', 'Ivy')
('female', 'male', 'Trudy')
('female', 'male', 'Margalo')
('male', 'female', 'Andre')
('male', 'female', 'Benji')
('male', 'female', 'Martainn')
('female', 'male', 'Pris')
('female', 'male', 'Trish')
('female', 'male', 'Gillan')
('female', 'male', 'Lindsay')


From the dev-test errors / mistakes, it looks like the model has trouble with names that can be used for both genders or names that don’t follow common patterns. It seems like it relies a lot on endings, which doesn’t always work.

In [22]:
# show the top features that are most useful for predicting gender / understanding model decisions

model_3.show_most_informative_features(20)

Most Informative Features
                last_two = 'na'           female : male   =     93.9 : 1.0
                last_two = 'la'           female : male   =     70.8 : 1.0
              last_three = 'ard'            male : female =     43.4 : 1.0
                last_two = 'rd'             male : female =     38.7 : 1.0
                last_two = 'ia'           female : male   =     35.9 : 1.0
                last_two = 'sa'           female : male   =     32.7 : 1.0
           ending_letter = 'a'            female : male   =     31.8 : 1.0
                last_two = 'ta'           female : male   =     29.3 : 1.0
              last_three = 'nne'          female : male   =     29.1 : 1.0
           ending_letter = 'k'              male : female =     28.6 : 1.0
                last_two = 'us'             male : female =     27.6 : 1.0
           ending_letter = 'f'              male : female =     26.8 : 1.0
                last_two = 'io'             male : female =     26.3 : 1.0

The most informative features show that the model relies heavily on the endings of names to predict gender. For example, names ending in “a,” “na,” or “la” are strongly associated with female names, while endings like “rd,” “us,” and “k” are more associated with male names.

**Conclusion**

Based on the analysis overall, the classifier improved as more features were added. Starting from a simple baseline, each version performed better by using more information from the names. The final model achieved the highest accuracy, although there was a slight drop on the test set, which is expected since it was completely unseen data. The results show that name endings are very important for predicting gender, but the model still struggles with gender-neutral or uncommon names.